## Section 1: Clean Setup & HF Authentication
Run this section to prepare your environment and log in to Hugging Face.

In [1]:
# 1. Install necessary libraries (clean environment)
!pip uninstall -y -q autoawq awq transformers accelerate bitsandbytes optimum auto-gptq gptqmodel
!pip install -q "torch" "torchvision" "torchaudio"
!pip install -q "transformers==4.51.3" "accelerate" "huggingface_hub" "safetensors" "sentencepiece" "protobuf"
!pip install -q "autoawq" "bitsandbytes" "nvidia-ml-py3" "pandas" "matplotlib" "seaborn" "datasets" "peft" "optimum"

# 2. Authenticate with Hugging Face (Programmatic login via HF_TOKEN)
from huggingface_hub import login
from google.colab import userdata

print("Logging in to Hugging Face...")
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged in!")
except userdata.SecretNotFoundError:
    print("Warning: 'HF_TOKEN' not found in Colab secrets. Please add it to access gated models.")
except Exception as e:
    print(f"An error occurred during login: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 18.7 MB/s eta 0:00:00
Logging in to Hugging Face...
Successfully logged in!


## Section 2: Data Loading & Perplexity Logic
This section loads the Wikitext dataset and defines the functions needed to evaluate model perplexity and throughput. You must run this before the batching section.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from tqdm import tqdm
import math
import time
import gc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("Loading wikitext-2-raw-v1 dataset...")
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
num_samples = 1000
if len(dataset) > num_samples:
    dataset = dataset.select(range(num_samples))
print(f"Using {len(dataset)} samples from the dataset.")

def calculate_perplexity_batched(model, tokenizer, dataset, device, batch_size=8, max_seq_len=512):
    model.eval()
    text = "\n\n".join(dataset["text"])
    encodings = tokenizer(text, return_tensors="pt", truncation=False)
    input_ids = encodings.input_ids[0]

    seq_len = input_ids.size(0)
    chunks = []
    for i in range(0, seq_len, max_seq_len):
        chunk = input_ids[i:i + max_seq_len]
        if chunk.size(0) == max_seq_len:
            chunks.append(chunk)

    if not chunks: return 0, 0
    stacked_inputs = torch.stack(chunks)
    num_chunks = stacked_inputs.size(0)

    total_loss = 0
    num_tokens = 0
    start_time = time.time()

    for i in tqdm(range(0, num_chunks, batch_size), desc="Calculating batched perplexity"):
        batch_inputs = stacked_inputs[i:i+batch_size].to(device)
        batch_labels = batch_inputs.clone()
        with torch.no_grad():
            outputs = model(batch_inputs, labels=batch_labels)
            loss = outputs.loss
            total_loss += loss.item() * batch_inputs.numel()
            num_tokens += batch_inputs.numel()

    duration = time.time() - start_time
    avg_nll = total_loss / num_tokens
    perplexity = math.exp(avg_nll)
    throughput = num_tokens / duration if duration > 0 else 0
    return perplexity, throughput

print("Perplexity logic defined successfully!")

Loading wikitext-2-raw-v1 dataset...


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Using 1000 samples from the dataset.
Perplexity logic defined successfully!


## Section 3: Batching Models & Graphing
This section tests FP16, INT8, and AWQ INT4 models using the batched logic, logs their metrics, and generates a clean comparative graph.

In [3]:
import gc
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import pynvml
import threading
import time
import os
import glob

pynvml.nvmlInit()
device_count = pynvml.nvmlDeviceGetCount()

class PowerMonitor(threading.Thread):
    def __init__(self, device_count):
        super().__init__()
        self.device_count = device_count
        self.handles = []
        for i in range(self.device_count):
            try:
                self.handles.append(pynvml.nvmlDeviceGetHandleByIndex(i))
            except Exception:
                print(f"Warning: Could not get NVML handle for device {i}. Skipping.")
        self.stop_event = threading.Event()
        self.power_readings = []

    def run(self):
        while not self.stop_event.is_set():
            try:
                current_power_sum = 0
                for h in self.handles:
                    current_power_sum += pynvml.nvmlDeviceGetPowerUsage(h) / 1000.0
                self.power_readings.append(current_power_sum)
            except Exception:
                pass
            time.sleep(0.1)

    def stop(self):
        self.stop_event.set()

configs = [
    {"name": "Llama-3.1-8B (FP16)", "id": "NousResearch/Meta-Llama-3.1-8B", "type": "fp16"},
    {"name": "Llama-3.1-8B (INT8 bnb)", "id": "NousResearch/Meta-Llama-3.1-8B", "type": "int8"},
    {"name": "Llama-3.1-8B (INT4 bnb)", "id": "NousResearch/Meta-Llama-3.1-8B", "type": "int4"},
    {"name": "Llama-3.1-8B (AWQ INT4)", "id": "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4", "type": "awq"},
    {"name": "Llama-3.2-1B (FP16)", "id": "meta-llama/Llama-3.2-1B-Instruct", "type": "fp16"},
    {"name": "Llama-3.2-1B (INT8 bnb)", "id": "meta-llama/Llama-3.2-1B-Instruct", "type": "int8"},
    {"name": "Llama-3.2-1B (INT4 bnb)", "id": "meta-llama/Llama-3.2-1B-Instruct", "type": "int4"},
    {"name": "Qwen 1.5-1.8B (FP16)", "id": "Qwen/Qwen1.5-1.8B-Chat", "type": "fp16"},
    {"name": "Qwen 1.5-1.8B (INT8 bnb)", "id": "Qwen/Qwen1.5-1.8B-Chat", "type": "int8"},
    {"name": "Qwen 1.5-1.8B (INT4 bnb)", "id": "Qwen/Qwen1.5-1.8B-Chat", "type": "int4"}
]

batch_sizes = [1, 8, 16]

RESULTS_DIR = "./llm_benchmarking_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

all_existing_dfs = []
existing_results_files = glob.glob(os.path.join(RESULTS_DIR, "batch_results_*.csv"))
for f in existing_results_files:
    try:
        all_existing_dfs.append(pd.read_csv(f))
    except Exception as e:
        print(f"Warning: Could not load existing results from {f}: {e}")

if all_existing_dfs:
    results_list = pd.concat(all_existing_dfs, ignore_index=True).to_dict('records')
    existing_df = pd.DataFrame(results_list).drop_duplicates(subset=['Model', 'Batch Size'], keep='last')
    results_list = existing_df.to_dict('records')
    print(f"Loaded {len(results_list)} existing results.")
else:
    results_list = []
    print("No existing results found, starting fresh.")

for config in configs:
    print(f"\n=== Evaluating {config['name']} ===")
    tokenizer = AutoTokenizer.from_pretrained(config['id'])
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model = None
    try:
        if any(r['Model'] == config['name'] and r['Batch Size'] in batch_sizes for r in results_list):
            print(f"  -> Configuration {config['name']} already in results. Skipping.")
            continue

        if config["type"] == "fp16":
            model = AutoModelForCausalLM.from_pretrained(config['id'], torch_dtype=torch.float16, device_map="auto")
        elif config["type"] == "int8":
            model = AutoModelForCausalLM.from_pretrained(config['id'], quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map="auto")
        elif config["type"] == "int4":
            model = AutoModelForCausalLM.from_pretrained(config['id'], quantization_config=BitsAndBytesConfig(load_in_4bit=True), device_map="auto")
        elif config["type"] == "awq":
            model = AutoModelForCausalLM.from_pretrained(config['id'], torch_dtype=torch.float16, device_map="auto")

        for bs in batch_sizes:
            if any(r['Model'] == config['name'] and r['Batch Size'] == bs for r in results_list):
                print(f"  -> Configuration {config['name']} with Batch Size {bs} already in results. Skipping.")
                continue

            print(f"  Testing Batch Size: {bs}")
            try:
                monitor = PowerMonitor(device_count)
                monitor.start()
                start_time = time.time()

                ppl, tp = calculate_perplexity_batched(model, tokenizer, dataset, model.device, batch_size=bs)

                duration = time.time() - start_time
                monitor.stop()
                monitor.join()

                peak_mem = torch.cuda.max_memory_allocated() / (1024**3)
                avg_power = np.mean(monitor.power_readings) if monitor.power_readings else 0
                total_energy = avg_power * duration

                results_list.append({
                    "Model": config["name"],
                    "Batch Size": bs,
                    "Perplexity": ppl,
                    "Throughput": tp,
                    "Peak Memory (GB)": peak_mem,
                    "Avg Power (W)": avg_power,
                    "Total Energy (J)": total_energy
                })
            except torch.cuda.OutOfMemoryError:
                print(f"  -> OOM at batch size {bs}. Clearing cache and trying next config.")
                if 'monitor' in locals() and monitor.is_alive():
                    monitor.stop()
                    monitor.join()
                torch.cuda.empty_cache()
            except Exception as e:
                print(f"  -> Error during batch processing for {config['name']} at batch size {bs}: {e}")
                if 'monitor' in locals() and monitor.is_alive():
                    monitor.stop()
                    monitor.join()
                torch.cuda.empty_cache()

    except torch.cuda.OutOfMemoryError:
        print(f"  -> OOM while loading model {config['name']}. Skipping this configuration.")
        torch.cuda.empty_cache()
        continue
    except Exception as e:
        print(f"  -> Failed to load or initialize {config['name']} due to: {e}. Skipping.")
        continue
    finally:
        if model is not None:
            del model
        gc.collect(); torch.cuda.empty_cache()

df_final = pd.DataFrame(results_list)
if not df_final.empty:
    output_filename = os.path.join(RESULTS_DIR, f"batch_results_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv")
    df_final.to_csv(output_filename, index=False)
    print(f"Results saved to {output_filename}")
display(df_final)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 18
})

fig, axes = plt.subplots(2, 3, figsize=(22, 12))
axes = axes.flatten()

metrics = [
    ("Throughput", "Throughput (tokens/sec)", "viridis", "higher"),
    ("Perplexity", "Perplexity", "magma", "lower"),
    ("Peak Memory (GB)", "Peak Memory (GB)", "coolwarm", "lower"),
    ("Avg Power (W)", "Average Power (W)", "plasma", "lower"),
    ("Total Energy (J)", "Total Energy (Joules)", "inferno", "lower")
]

for i, (col, ylabel, palette, direction) in enumerate(metrics):
    sns.barplot(data=df_final, x="Model", y=col, hue="Batch Size", ax=axes[i], palette=palette, edgecolor='black', linewidth=0.5)
    title = f"{col} Comparison" + (" (Lower is Better)" if direction == "lower" else " (Higher is Better)")
    axes[i].set_title(title, fontweight='bold')
    axes[i].set_ylabel(ylabel)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(axis='y', linestyle='--', alpha=0.7)
    if i == 0:
        axes[i].legend(title='Batch Size', bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    else:
        axes[i].legend().remove()

axes[-1].axis('off')
plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.show()

No existing results found, starting fresh.

=== Evaluating Llama-3.1-8B (FP16) ===


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

...

  Testing Batch Size: 16


Calculating batched perplexity: 100%|██████████| 9/9 [00:05<00:00,  1.52it/s]



=== Evaluating Llama-3.1-8B (INT8 bnb) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Section 4: Sparsity Evaluation
This section tests Dense FP16 against Naive 2:4 Sparsity and MaskLLM 2:4 Sparsity. It monitors Perplexity, Throughput, Peak Memory, Average Power, and Total Energy across batch sizes 1, 8, and 16.

In [ ]:
import gc
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download
import pynvml
import threading
import time
import os
import glob

# Ensure NVML is initialized
try:
    pynvml.nvmlInit()
    device_count = pynvml.nvmlDeviceGetCount()
except Exception as e:
    print(f"NVML init failed: {e}. Power monitoring will be skipped.")
    device_count = 0 # Indicate no devices for monitoring

# Redefining PowerMonitor for consistency
class PowerMonitor(threading.Thread):
    def __init__(self, device_count):
        super().__init__()
        self.device_count = device_count
        self.handles = []
        for i in range(self.device_count):
            try:
                self.handles.append(pynvml.nvmlDeviceGetHandleByIndex(i))
            except Exception:
                print(f"Warning: Could not get NVML handle for device {i}. Skipping.")
        self.stop_event = threading.Event()
        self.power_readings = []

    def run(self):
        while not self.stop_event.is_set():
            try:
                current_power_sum = 0
                for h in self.handles:
                    current_power_sum += pynvml.nvmlDeviceGetPowerUsage(h) / 1000.0
                self.power_readings.append(current_power_sum)
            except Exception:
                pass # Device might become unavailable or other NVML errors
            time.sleep(0.1)

    def stop(self):
        self.stop_event.set()

# Sparsity Helpers
def apply_2_4_sparsity(tensor):
    shape = tensor.shape
    w_4 = tensor.abs().reshape(-1, 4)
    _, idx = w_4.topk(2, dim=-1, largest=False)
    mask = torch.ones_like(w_4)
    mask.scatter_(1, idx, 0)
    return (tensor.reshape(-1, 4) * mask).reshape(shape)

def decompress_mask_bitmap(compressed_mask_uint8, target_shape):
    full_boolean_mask_np = np.unpackbits(compressed_mask_uint8, axis=None).astype(bool)
    return full_boolean_mask_np.reshape(target_shape)

# Fetch MaskLLM Masks
maskllm_repo_id = "Vinnnf/LLaMA-3.1-8B-MaskLLM-C4"
print(f"Downloading MaskLLM compressed masks from {maskllm_repo_id}...")
mask_path = hf_hub_download(repo_id=maskllm_repo_id, filename="mask_compressed.npz")
masks = np.load(mask_path)

configs_sparse = [
    {"name": "Llama-3.1-8B (Dense FP16 Baseline)", "id": "NousResearch/Meta-Llama-3.1-8B", "type": "dense"},
    {"name": "Llama-3.1-8B (Naive 2:4 Sparsity)", "id": "NousResearch/Meta-Llama-3.1-8B", "type": "naive_2_4"},
    {"name": "Llama-3.1-8B (MaskLLM 2:4)", "id": "NousResearch/Meta-Llama-3.1-8B", "type": "maskllm"},
    {"name": "Qwen 1.5-1.8B (Dense FP16 Baseline)", "id": "Qwen/Qwen1.5-1.8B-Chat", "type": "dense"},
    {"name": "Qwen 1.5-1.8B (Naive 2:4 Sparsity)", "id": "Qwen/Qwen1.5-1.8B-Chat", "type": "naive_2_4"},
    {"name": "Qwen 1.5-1.8B (MaskLLM 2:4)", "id": "Qwen/Qwen1.5-1.8B-Chat", "type": "maskllm"}
]

batch_sizes = [16]

RESULTS_DIR = "./llm_benchmarking_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load existing sparse results if any
all_existing_sparse_dfs = []
existing_sparse_files = glob.glob(os.path.join(RESULTS_DIR, "sparse_results_*.csv"))
for f in existing_sparse_files:
    try:
        all_existing_sparse_dfs.append(pd.read_csv(f))
    except Exception as e:
        print(f"Warning: Could not load existing sparse results from {f}: {e}")

if all_existing_sparse_dfs:
    results_sparse = pd.concat(all_existing_sparse_dfs, ignore_index=True).to_dict('records')
    # Remove duplicates based on Model and Batch Size, keeping the latest if any are duplicated
    existing_df_sparse = pd.DataFrame(results_sparse).drop_duplicates(subset=['Model', 'Batch Size'], keep='last')
    results_sparse = existing_df_sparse.to_dict('records')
    print(f"Loaded {len(results_sparse)} existing sparse results.")
else:
    results_sparse = []
    print("No existing sparse results found, starting fresh.")

for config in configs_sparse:
    print(f"\n=== Evaluating {config['name']} ===")
    tokenizer = AutoTokenizer.from_pretrained(config['id'])
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model = None # Initialize model for OOM handling
    try:
        # Check if this config has already been run successfully
        if any(r['Model'] == config['name'] and r['Batch Size'] in batch_sizes for r in results_sparse):
            print(f"  -> Configuration {config['name']} already in sparse results. Skipping.")
            continue

        model = AutoModelForCausalLM.from_pretrained(config['id'], torch_dtype=torch.float16, device_map="auto")

        # Apply sparsity if needed
        if config["type"] == "naive_2_4":
            print("  Applying naive magnitude-based 2:4 sparsity mask...")
            applied_layers = 0
            for name, module in model.named_modules():
                if isinstance(module, torch.nn.Linear) and 'lm_head' not in name:
                    sparse_weight = apply_2_4_sparsity(module.weight.data)
                    if hasattr(torch.sparse, 'to_sparse_semi_structured'):
                        try:
                            sparse_tensor = torch.sparse.to_sparse_semi_structured(sparse_weight)
                            module.weight = torch.nn.Parameter(sparse_tensor)
                        except Exception:
                            module.weight.data = sparse_weight
                    else:
                        module.weight.data = sparse_weight
                    applied_layers += 1
            print(f"  [Success] Applied Naive 2:4 Sparsity to {applied_layers} layers.")

        elif config["type"] == "maskllm":
            print("  Applying pre-trained MaskLLM 2:4 masks...")
            applied_layers = 0
            missing_masks = 0
            for name, module in model.named_modules():
                if isinstance(module, torch.nn.Linear) and 'lm_head' not in name:
                    mask_key = name + ".weight.mask"
                    if mask_key in masks:
                        mask_data_np_compressed = masks[mask_key]
                        target_shape = module.weight.data.shape
                        try:
                            full_mask_np = decompress_mask_bitmap(mask_data_np_compressed, target_shape)
                            mask_tensor = torch.tensor(full_mask_np, device=module.weight.device, dtype=module.weight.dtype)
                            sparse_weight = module.weight.data * mask_tensor
                            if hasattr(torch.sparse, 'to_sparse_semi_structured'):
                                try:
                                    sparse_tensor = torch.sparse.to_sparse_semi_structured(sparse_weight)
                                    module.weight = torch.nn.Parameter(sparse_tensor)
                                except Exception:
                                    module.weight.data = sparse_weight
                            else:
                                module.weight.data = sparse_weight
                            applied_layers += 1
                        except Exception as e:
                            print(f"  Warning: Error on {mask_key} - {e}")
                    else:
                        missing_masks += 1
            print(f"  [Success] Applied MaskLLM Sparsity to {applied_layers} layers. (Missing masks for {missing_masks} layers)")

        for bs in batch_sizes:
            # Check if this specific batch size for this model has been run
            if any(r['Model'] == config['name'] and r['Batch Size'] == bs for r in results_sparse):
                print(f"  -> Configuration {config['name']} with Batch Size {bs} already in sparse results. Skipping.")
                continue

            print(f"  Testing Batch Size: {bs}")
            try:
                monitor = PowerMonitor(device_count)
                monitor.start()
                start_time = time.time()

                ppl, tp = calculate_perplexity_batched(model, tokenizer, dataset, model.device, batch_size=bs)

                duration = time.time() - start_time
                monitor.stop()
                monitor.join()

                peak_mem = torch.cuda.max_memory_allocated() / (1024**3)
                avg_power = np.mean(monitor.power_readings) if monitor.power_readings else 0
                total_energy = avg_power * duration # Joules

                results_sparse.append({
                    "Model": config["name"],
                    "Batch Size": bs,
                    "Perplexity": ppl,
                    "Throughput": tp,
                    "Peak Memory (GB)": peak_mem,
                    "Avg Power (W)": avg_power,
                    "Total Energy (J)": total_energy
                })
            except torch.cuda.OutOfMemoryError:
                print(f"  -> OOM at batch size {bs}. Clearing cache and trying next batch size/config.")
                if 'monitor' in locals() and monitor.is_alive():
                    monitor.stop()
                    monitor.join()
                torch.cuda.empty_cache()
            except Exception as e:
                print(f"  -> Error during batch processing for {config['name']} at batch size {bs}: {e}")
                if 'monitor' in locals() and monitor.is_alive():
                    monitor.stop()
                    monitor.join()
                torch.cuda.empty_cache()

    except torch.cuda.OutOfMemoryError:
        print(f"  -> OOM while loading model {config['name']}. Skipping this configuration.")
        torch.cuda.empty_cache()
        continue # Skip to the next configuration
    except Exception as e:
        print(f"  -> Failed to load or initialize {config['name']} due to: {e}. Skipping this configuration.")
        continue # Skip to the next configuration

    finally:
        if model is not None:
            del model
        gc.collect(); torch.cuda.empty_cache()

# Display Data and save to file
df_sparse = pd.DataFrame(results_sparse)
if not df_sparse.empty:
    output_filename_sparse = os.path.join(RESULTS_DIR, f"sparse_results_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv")
    df_sparse.to_csv(output_filename_sparse, index=False)
    print(f"Sparse results saved to {output_filename_sparse}")
display(df_sparse)

# Graphing


plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 18
})

fig, axes = plt.subplots(2, 3, figsize=(22, 12))
axes = axes.flatten()

metrics = [
    ("Throughput", "Throughput (tokens/sec)", "viridis", "higher"),
    ("Perplexity", "Perplexity", "magma", "lower"),
    ("Peak Memory (GB)", "Peak Memory (GB)", "coolwarm", "lower"),
    ("Avg Power (W)", "Average Power (W)", "plasma", "lower"),
    ("Total Energy (J)", "Total Energy (Joules)", "inferno", "lower")
]

for i, (col, ylabel, palette, direction) in enumerate(metrics):
    sns.barplot(data=df_sparse, x="Model", y=col, ax=axes[i], palette=palette, edgecolor='black', linewidth=0.5)
    title = f"{col} Comparison" + (" (Lower is Better)" if direction == "lower" else " (Higher is Better)")
    axes[i].set_title(title, fontweight='bold')
    axes[i].set_ylabel(ylabel)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(axis='y', linestyle='--', alpha=0.7)

    if len(df_sparse['Model'].unique()) > 1:
        axes[i].legend().remove()

# Hide the 6th empty subplot
axes[-1].axis('off')

plt.tight_layout(rect=[0, 0, 0.95, 1]) # Adjust layout to make space for legend (if any)
plt.show()

## Section 5: Pareto Plot (Memory vs. Throughput)
This scatter plot visualizes the trade-offs between memory footprint and text generation throughput. The size of each point indicates **Energy Efficiency** (Tokens per Joule).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as pd
import pandas as pd
import os
import glob

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 18
})

RESULTS_DIR = "./llm_benchmarking_results"
all_results_files = glob.glob(os.path.join(RESULTS_DIR, "*.csv"))

if not all_results_files:
    print(f"No results files found in {RESULTS_DIR} to plot. Please run Section 3 and Section 4 first.")
else:
    all_dfs = []
    for f in all_results_files:
        try:
            all_dfs.append(pd.read_csv(f))
        except Exception as e:
            print(f"Warning: Could not load results from {f}: {e}")

    if not all_dfs:
        print("No valid results dataframes could be loaded.")
    else:
        df_final = pd.concat(all_dfs, ignore_index=True)
        df_final.drop_duplicates(subset=['Model', 'Batch Size'], inplace=True)

        df_final['Energy Efficiency (Tokens/Joule)'] = df_final.apply(lambda row: row['Throughput'] / row['Avg Power (W)'] if row['Avg Power (W)'] != 0 else 0, axis=1)
        df_pareto = df_final[df_final['Model'].str.contains('8B')].copy()

        plt.figure(figsize=(14, 9))

        sns.scatterplot(
            data=df_pareto,
            x='Peak Memory (GB)',
            y='Throughput',
            hue='Model',
            size='Energy Efficiency (Tokens/Joule)',
            sizes=(150, 1000),
            alpha=0.8,
            palette='tab10',
            edgecolor='black',
            linewidth=0.5
        )

        for i in range(df_pareto.shape[0]):
            plt.text(
                df_pareto['Peak Memory (GB)'].iloc[i] + 0.3,
                df_pareto['Throughput'].iloc[i],
                f"BS:{df_pareto['Batch Size'].iloc[i]}",
                fontsize=11,
                alpha=0.9,
                verticalalignment='center',
                horizontalalignment='left'
            )

        plt.title('Pareto Front: Peak Memory vs. Throughput\n(Marker Size = Energy Efficiency)', fontweight='bold')
        plt.xlabel('Peak Memory (GB) [Lower is Better]')
        plt.ylabel('Throughput (tokens/sec) [Higher is Better]')

        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., title='Model & Efficiency')
        plt.tight_layout(rect=[0, 0, 0.88, 1])
        plt.show()

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

SOURCE_DIR = "./llm_benchmarking_results"
DESTINATION_DIR = "/content/drive/MyDrive/llm_benchmarking_data"

!mkdir -p "{DESTINATION_DIR}"

print(f"Copying results from {SOURCE_DIR} to {DESTINATION_DIR}...")
!rsync -av --exclude='.*' "{SOURCE_DIR}/" "{DESTINATION_DIR}/"

print(f"Results successfully copied to Google Drive: {DESTINATION_DIR}")
print("You can verify the contents in your Google Drive.")